## **Setup dan Instalasi Awal**

In [ ]:
# Step 1: Import Libraries and Mount Google Drive
import os
import shutil
import pandas as pd
import random
from tqdm import tqdm
from google.colab import drive

drive.mount("/content/gdrive")

# Step 2: Clone YOLOv5 Repository and Install Dependencies
!git clone https://github.com/ultralytics/yolov5.git
!cp -r /content/yolov5 /content/gdrive/MyDrive/
%cd /content/gdrive/MyDrive/yolov5
!pip install -U -r requirements.txt
!pip install "numpy<2"

Mounted at /content/gdrive
Cloning into 'yolov5'...
remote: Enumerating objects: 17075, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 17075 (delta 19), reused 7 (delta 7), pack-reused 17049 (from 2)
Receiving objects: 100% (17075/17075), 15.68 MiB | 20.99 MiB/s, done.
Resolving deltas: 100% (11719/11719), done.
/content/gdrive/MyDrive/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 533.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.1
    Uninstalling numpy-2.2.1:
      Successfully uninstalled numpy-2.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.10.1 requires pandas<2.2.3dev0,>=2.0, but you have pandas 2.2.3 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.14.1 which is incompatible.


In [ ]:
!pwd

/content/gdrive/MyDrive/yolov5


# **Pengaturan Dataset dan Pembagian Data**

In [ ]:
import os
import shutil
import random

# Path asli dataset
base_dir = "/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train"
output_dir = "/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train-output"  # Folder output

# Folder output untuk gambar dan label
train_image_dir = os.path.join(output_dir, "images/train")
valid_image_dir = os.path.join(output_dir, "images/val")
train_label_dir = os.path.join(output_dir, "labels/train")
valid_label_dir = os.path.join(output_dir, "labels/val")

# Buat folder output jika belum ada
for folder in [train_image_dir, valid_image_dir, train_label_dir, valid_label_dir]:
    os.makedirs(folder, exist_ok=True)

# Rasio split train-valid
train_ratio = 0.8

# Loop untuk membaca dataset
for subject in os.listdir(base_dir):
    subject_path = os.path.join(base_dir, subject)
    if os.path.isdir(subject_path):  # Pastikan subjek adalah folder
        for category in ["fall", "non_fall"]:
            category_path = os.path.join(subject_path, category)
            if os.path.isdir(category_path):
                label = 1 if category == "fall" else 0  # Label: 1 untuk fall, 0 untuk non_fall
                for activity in os.listdir(category_path):
                    activity_path = os.path.join(category_path, activity)
                    if os.path.isdir(activity_path):
                        for frame in os.listdir(activity_path):
                            if frame.endswith(".jpg"):  # Hanya memproses file gambar
                                src_image = os.path.join(activity_path, frame)

                                # Random split untuk train atau valid
                                if random.random() < train_ratio:
                                    image_dir = train_image_dir
                                    label_dir = train_label_dir
                                else:
                                    image_dir = valid_image_dir
                                    label_dir = valid_label_dir

                                # Penamaan file baru
                                file_name = f"{subject}_{category}_{activity}_{frame}"
                                dest_image = os.path.join(image_dir, file_name)

                                # Salin gambar ke folder yang sesuai
                                shutil.copy(src_image, dest_image)

                                # Buat file label
                                label_file = os.path.join(label_dir, file_name.replace('.jpg', '.txt'))
                                with open(label_file, "w") as f:
                                    f.write(f"{label} 0.5 0.5 1.0 1.0\n")

# Sinkronisasi akhir: Periksa gambar tanpa label dan label tanpa gambar
for img_dir, lbl_dir in [(train_image_dir, train_label_dir), (valid_image_dir, valid_label_dir)]:
    img_files = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith('.jpg')}
    lbl_files = {os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith('.txt')}

    # Cari gambar tanpa label
    missing_labels = img_files - lbl_files
    for img_name in missing_labels:
        label_file = os.path.join(lbl_dir, f"{img_name}.txt")
        with open(label_file, "w") as f:
            # Buat label default (tanpa anotasi)
            f.write(f"0 0.5 0.5 1.0 1.0\n")

print("Dataset telah diproses dan disusun ulang.")

Dataset telah diproses dan disusun ulang.


## **Konfigurasi Data dalam Format YAML**

In [ ]:
import yaml

# Tentukan data untuk file YAML
data = {
    'train': '/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train-output/images/train',
    'val': '/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train-output/images/val',
    'nc': 2,  # Jumlah kelas
    'names': {0: 'non_fall', 1: 'fall'}
}

# Tentukan path untuk menyimpan file YAML
yaml_file = '/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml'

# Menulis data ke dalam file YAML
with open(yaml_file, 'w') as file:
    yaml.dump(data, file)

print(f'File YAML telah dibuat di: {yaml_file}')


File YAML telah dibuat di: /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml


## **Pelatihan Model**

In [ ]:
!python train.py --img 640 --batch 16 --epochs 25 --data /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml --weights yolov5s.pt


Streaming output truncated to the last 5000 lines.
  with torch.cuda.amp.autocast(amp):
      15/24      3.58G   0.004254   0.002706    0.01349         52        640:  50% 131/260 [01:43<01:48,  1.19it/s]/content/gdrive/MyDrive/yolov5/train.py:412: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      15/24      3.58G   0.004253   0.002704    0.01349         51        640:  51% 132/260 [01:43<01:28,  1.45it/s]/content/gdrive/MyDrive/yolov5/train.py:412: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      15/24      3.58G   0.004247   0.002705    0.01349         54        640:  51% 133/260 [01:44<02:03,  1.03it/s]/content/gdrive/MyDrive/yolov5/train.py:412: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args.

## **Evaluasi Model**

In [ ]:
!python val.py --data /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml --weights /content/gdrive/MyDrive/yolov5/runs/train/exp/weights/best.pt --batch-size 16

val: data=/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml, weights=['/content/gdrive/MyDrive/yolov5/runs/train/exp/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-389-ge62a31b6 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train-output/labels/val... 2867 images, 0 backgrounds, 0 corrupt: 100% 2867/2867 [12:34<00:00,  3.80it/s]
val: New cache created: /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/train-output/labels/val.cache
                 Class     Images  Instances       

## **Cek Semua Model yang Tersimpan**

In [ ]:
import os
import glob

# Cari semua folder experiment
base_path = "/content/gdrive/MyDrive/yolov5/runs/train"
experiments = sorted([d for d in os.listdir(base_path) if d.startswith("exp")])

print("📁 Daftar Model yang Tersimpan:\n")
for exp in experiments:
    weights_path = os.path.join(base_path, exp, "weights")
    if os.path.exists(weights_path):
        best_pt = os.path.join(weights_path, "best.pt")
        last_pt = os.path.join(weights_path, "last.pt")
        
        print(f"📦 {exp}/")
        if os.path.exists(best_pt):
            size_mb = os.path.getsize(best_pt) / (1024*1024)
            print(f"   ✅ best.pt ({size_mb:.1f} MB)")
        if os.path.exists(last_pt):
            size_mb = os.path.getsize(last_pt) / (1024*1024)
            print(f"   ✅ last.pt ({size_mb:.1f} MB)")
        
        # Cek results.txt untuk melihat metrics
        results_file = os.path.join(base_path, exp, "results.txt")
        if os.path.exists(results_file):
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) > 1:
                    # Last epoch metrics
                    last_line = lines[-1].strip().split()
                    if len(last_line) >= 10:
                        print(f"   📊 mAP@0.5: {last_line[7]}")
                        print(f"   📊 mAP@0.5:0.95: {last_line[8]}")
        print()

## **Evaluasi Semua Model untuk Perbandingan**

In [ ]:
# ============================================================================
# ANALISIS MODEL EXP2 LENGKAP
# ============================================================================

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
import os

print("="*70)
print("📊 ANALISIS MODEL EXP2")
print("="*70)

# Path ke exp2
exp2_path = "/content/gdrive/MyDrive/yolov5/runs/train/exp2"
results_csv = os.path.join(exp2_path, "results.csv")

# ============================================================================
# 1. CEK TRAINING METRICS DARI results.csv
# ============================================================================

if os.path.exists(results_csv):
    print("\n✅ results.csv ditemukan!")
    
    # Baca CSV
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Clean whitespace
    
    print(f"\n📈 Total Epochs: {len(df)}")
    print(f"📈 Columns: {list(df.columns)}")
    
    # Last epoch metrics
    last_epoch = df.iloc[-1]
    
    print("\n" + "="*70)
    print("🎯 FINAL METRICS (Last Epoch)")
    print("="*70)
    
    # Check available columns and print
    if 'metrics/precision(B)' in df.columns:
        precision = last_epoch['metrics/precision(B)']
        print(f"📊 Precision: {precision:.4f}")
    
    if 'metrics/recall(B)' in df.columns:
        recall = last_epoch['metrics/recall(B)']
        print(f"📊 Recall: {recall:.4f}")
    
    if 'metrics/mAP_0.5(B)' in df.columns:
        map50 = last_epoch['metrics/mAP_0.5(B)']
        print(f"📊 mAP@0.5: {map50:.4f}")
        
        # DECISION LOGIC
        if map50 > 0.85:
            print("\n🏆 MODEL EXCELLENT! mAP > 0.85")
        elif map50 > 0.70:
            print("\n✅ MODEL GOOD! mAP > 0.70")
        elif map50 > 0.50:
            print("\n⚠️  MODEL ACCEPTABLE, tapi bisa lebih baik")
        else:
            print("\n❌ MODEL BURUK! mAP < 0.50")
    
    if 'metrics/mAP_0.5:0.95(B)' in df.columns:
        map5095 = last_epoch['metrics/mAP_0.5:0.95(B)']
        print(f"📊 mAP@0.5:0.95: {map5095:.4f}")
    
    # Plot training curves
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('EXP2 Training Metrics', fontsize=16, fontweight='bold')
    
    # Precision
    if 'metrics/precision(B)' in df.columns:
        axes[0,0].plot(df['metrics/precision(B)'], linewidth=2, color='blue')
        axes[0,0].set_title('Precision')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].grid(True, alpha=0.3)
        axes[0,0].axhline(y=0.85, color='green', linestyle='--', alpha=0.5, label='Excellent')
        axes[0,0].axhline(y=0.70, color='orange', linestyle='--', alpha=0.5, label='Good')
        axes[0,0].legend()
    
    # Recall
    if 'metrics/recall(B)' in df.columns:
        axes[0,1].plot(df['metrics/recall(B)'], linewidth=2, color='orange')
        axes[0,1].set_title('Recall')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].grid(True, alpha=0.3)
        axes[0,1].axhline(y=0.85, color='green', linestyle='--', alpha=0.5)
        axes[0,1].axhline(y=0.70, color='orange', linestyle='--', alpha=0.5)
    
    # mAP@0.5
    if 'metrics/mAP_0.5(B)' in df.columns:
        axes[0,2].plot(df['metrics/mAP_0.5(B)'], linewidth=2, color='green')
        axes[0,2].set_title('mAP@0.5 (MOST IMPORTANT!)')
        axes[0,2].set_xlabel('Epoch')
        axes[0,2].grid(True, alpha=0.3)
        axes[0,2].axhline(y=0.85, color='green', linestyle='--', alpha=0.5)
        axes[0,2].axhline(y=0.70, color='orange', linestyle='--', alpha=0.5)
    
    # Box Loss
    if 'train/box_loss' in df.columns:
        axes[1,0].plot(df['train/box_loss'], linewidth=2, color='red', label='Train')
        axes[1,0].set_title('Box Loss')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].grid(True, alpha=0.3)
        axes[1,0].legend()
    
    # Obj Loss
    if 'train/obj_loss' in df.columns:
        axes[1,1].plot(df['train/obj_loss'], linewidth=2, color='purple', label='Train')
        axes[1,1].set_title('Objectness Loss')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].grid(True, alpha=0.3)
        axes[1,1].legend()
    
    # Class Loss
    if 'train/cls_loss' in df.columns:
        axes[1,2].plot(df['train/cls_loss'], linewidth=2, color='brown', label='Train')
        axes[1,2].set_title('Classification Loss')
        axes[1,2].set_xlabel('Epoch')
        axes[1,2].grid(True, alpha=0.3)
        axes[1,2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print full DataFrame
    print("\n" + "="*70)
    print("📋 FULL TRAINING LOG (Last 5 Epochs)")
    print("="*70)
    print(df.tail().to_string())
    
else:
    print("\n❌ results.csv tidak ditemukan di exp2!")

# ============================================================================
# 2. TAMPILKAN VISUALISASI DATASET
# ============================================================================

print("\n" + "="*70)
print("📊 DATASET ANALYSIS")
print("="*70)

# Labels distribution
labels_jpg = os.path.join(exp2_path, "labels.jpg")
if os.path.exists(labels_jpg):
    print("\n📊 Labels Distribution:")
    display(Image(filename=labels_jpg, width=800))
else:
    print("\n⚠️  labels.jpg tidak ditemukan")

# Labels correlogram
labels_corr = os.path.join(exp2_path, "labels_correlogram.jpg")
if os.path.exists(labels_corr):
    print("\n📊 Labels Correlogram:")
    display(Image(filename=labels_corr, width=600))
else:
    print("\n⚠️  labels_correlogram.jpg tidak ditemukan")

# Training batches
print("\n📊 Training Batch Samples:")
for i in range(3):
    batch_img = os.path.join(exp2_path, f"train_batch{i}.jpg")
    if os.path.exists(batch_img):
        print(f"\nBatch {i}:")
        display(Image(filename=batch_img, width=800))

print("\n" + "="*70)
print("💡 NEXT STEP: Run validation untuk generate F1-curve!")
print("="*70)

## **🎯 GENERATE F1-CURVE untuk EXP2 (PENTING!)**

In [ ]:
# RUN VALIDATION untuk generate F1-curve dan metrics lengkap
print("🔄 Running validation on exp2/best.pt...")
print("This will generate:")
print("  - F1-Confidence Curve")
print("  - Precision-Recall Curve")
print("  - Confusion Matrix")
print("  - Per-class metrics")
print("\n" + "="*70)

!python val.py \
  --data /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml \
  --weights /content/gdrive/MyDrive/yolov5/runs/train/exp2/weights/best.pt \
  --batch-size 16 \
  --save-json \
  --save-txt \
  --verbose \
  --plots

print("\n" + "="*70)
print("✅ Validation complete!")
print("="*70)

## **📊 HASIL VALIDASI EXP2 - F1 CURVE & METRICS**

In [ ]:
from IPython.display import Image, display
import os

# Path ke hasil validation (biasanya di runs/val/exp atau exp2)
val_path = "/content/gdrive/MyDrive/yolov5/runs/val"

# Cari folder validation terbaru
val_folders = sorted([d for d in os.listdir(val_path) if d.startswith("exp")])
if val_folders:
    latest_val = val_folders[-1]  # exp terbaru
    val_results_path = os.path.join(val_path, latest_val)
    
    print("="*70)
    print(f"📊 VALIDATION RESULTS: {latest_val}")
    print("="*70)
    
    # 1. F1-Confidence Curve (PALING PENTING!)
    f1_curve = os.path.join(val_results_path, "F1_curve.png")
    if os.path.exists(f1_curve):
        print("\n🎯 F1-CONFIDENCE CURVE:")
        print("Ini adalah metrics PALING PENTING untuk menilai model!")
        print("Yang perlu dicari:")
        print("  ✅ F1 (non_fall) > 0.70")
        print("  ✅ F1 (fall) > 0.70")
        print("  ✅ All classes F1 > 0.70")
        display(Image(filename=f1_curve, width=900))
    
    # 2. Precision-Recall Curve
    pr_curve = os.path.join(val_results_path, "PR_curve.png")
    if os.path.exists(pr_curve):
        print("\n📈 PRECISION-RECALL CURVE:")
        display(Image(filename=pr_curve, width=900))
    
    # 3. Confusion Matrix
    confusion = os.path.join(val_results_path, "confusion_matrix.png")
    if os.path.exists(confusion):
        print("\n📊 CONFUSION MATRIX:")
        print("Yang bagus:")
        print("  ✅ Diagonal values tinggi (prediksi benar)")
        print("  ✅ Off-diagonal values rendah (prediksi salah)")
        display(Image(filename=confusion, width=700))
    
    # 4. Results summary
    results_txt = os.path.join(val_results_path, "results.txt")
    if os.path.exists(results_txt):
        print("\n" + "="*70)
        print("📋 DETAILED METRICS:")
        print("="*70)
        with open(results_txt, 'r') as f:
            print(f.read())
    
    # DECISION LOGIC
    print("\n" + "="*70)
    print("🎯 KEPUTUSAN: MODEL EXP2 BAGUS ATAU TIDAK?")
    print("="*70)
    print("\nLihat F1-Confidence Curve di atas:")
    print("\n✅ MODEL BAGUS jika:")
    print("   - F1 (non_fall) > 0.70 (garis biru)")
    print("   - F1 (fall) > 0.70 (garis orange)")
    print("   - mAP@0.5 > 0.85 (excellent) atau > 0.70 (good)")
    print("\n❌ MODEL BURUK jika:")
    print("   - Salah satu class F1 < 0.50")
    print("   - Ada class yang hampir 0 (seperti exp)")
    print("   - mAP@0.5 < 0.70")
    print("\n💡 Bandingkan dengan exp (F1=0.56) di cell sebelumnya!")
    
else:
    print("❌ Folder validation tidak ditemukan!")
    print("Jalankan cell sebelumnya dulu untuk validasi.")

## **📊 Bandingkan Semua Experiment (exp vs exp2 vs exp3)**

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

base_path = "/content/gdrive/MyDrive/yolov5/runs/train"
experiments = ["exp", "exp2", "exp3", "exp4", "exp5"]  # Tambahkan semua yang mungkin ada

comparison_data = []

print("="*70)
print("🏆 MODEL COMPARISON - FINAL METRICS")
print("="*70)

for exp in experiments:
    results_file = os.path.join(base_path, exp, "results.txt")
    weights_file = os.path.join(base_path, exp, "weights", "best.pt")
    
    if os.path.exists(results_file) and os.path.exists(weights_file):
        try:
            # Read results
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) > 1:
                    # Parse last line
                    headers = lines[0].strip().split()
                    values = lines[-1].strip().split()
                    
                    # Get key metrics (columns might vary)
                    try:
                        precision_idx = headers.index('metrics/precision(B)') if 'metrics/precision(B)' in headers else 6
                        recall_idx = headers.index('metrics/recall(B)') if 'metrics/recall(B)' in headers else 7
                        map50_idx = headers.index('metrics/mAP_0.5(B)') if 'metrics/mAP_0.5(B)' in headers else 8
                        map5095_idx = headers.index('metrics/mAP_0.5:0.95(B)') if 'metrics/mAP_0.5:0.95(B)' in headers else 9
                        
                        precision = float(values[precision_idx]) if len(values) > precision_idx else 0
                        recall = float(values[recall_idx]) if len(values) > recall_idx else 0
                        map50 = float(values[map50_idx]) if len(values) > map50_idx else 0
                        map5095 = float(values[map5095_idx]) if len(values) > map5095_idx else 0
                        
                        # Calculate F1 score
                        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                        
                        # Get file size
                        size_mb = os.path.getsize(weights_file) / (1024*1024)
                        
                        comparison_data.append({
                            'Experiment': exp,
                            'Precision': precision,
                            'Recall': recall,
                            'F1-Score': f1,
                            'mAP@0.5': map50,
                            'mAP@0.5:0.95': map5095,
                            'Size (MB)': size_mb
                        })
                        
                        # Print individual results
                        status = "🏆" if map50 > 0.85 else "✅" if map50 > 0.70 else "⚠️" if map50 > 0.50 else "❌"
                        print(f"\n{status} {exp.upper()}")
                        print(f"   Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
                        print(f"   mAP@0.5: {map50:.4f} | mAP@0.5:0.95: {map5095:.4f}")
                        print(f"   Model Size: {size_mb:.1f} MB")
                        
                    except Exception as e:
                        print(f"⚠️ {exp}: Error parsing metrics - {e}")
        except Exception as e:
            print(f"⚠️ {exp}: Error reading file - {e}")

if comparison_data:
    # Create comparison DataFrame
    df_compare = pd.DataFrame(comparison_data)
    df_compare = df_compare.sort_values('mAP@0.5', ascending=False)
    
    print("\n" + "="*70)
    print("📊 RANKING BY mAP@0.5")
    print("="*70)
    print(df_compare.to_string(index=False))
    
    # Find best model
    best_model = df_compare.iloc[0]
    print("\n" + "="*70)
    print(f"🥇 BEST MODEL: {best_model['Experiment'].upper()}")
    print("="*70)
    print(f"✅ Precision: {best_model['Precision']:.4f}")
    print(f"✅ Recall: {best_model['Recall']:.4f}")
    print(f"✅ F1-Score: {best_model['F1-Score']:.4f}")
    print(f"✅ mAP@0.5: {best_model['mAP@0.5']:.4f}")
    print(f"✅ mAP@0.5:0.95: {best_model['mAP@0.5:0.95']:.4f}")
    print(f"\n📂 Path: /content/gdrive/MyDrive/yolov5/runs/train/{best_model['Experiment']}/weights/best.pt")
    print(f"💾 Size: {best_model['Size (MB)']:.1f} MB")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].barh(df_compare['Experiment'], df_compare['mAP@0.5'], color='skyblue')
    axes[0].set_xlabel('mAP@0.5')
    axes[0].set_title('Model Comparison: mAP@0.5')
    axes[0].axvline(x=0.85, color='green', linestyle='--', label='Excellent (>0.85)')
    axes[0].axvline(x=0.70, color='orange', linestyle='--', label='Good (>0.70)')
    axes[0].legend()
    
    axes[1].barh(df_compare['Experiment'], df_compare['F1-Score'], color='lightcoral')
    axes[1].set_xlabel('F1-Score')
    axes[1].set_title('Model Comparison: F1-Score')
    
    axes[2].barh(df_compare['Experiment'], df_compare['Precision'], color='lightgreen', alpha=0.7, label='Precision')
    axes[2].barh(df_compare['Experiment'], df_compare['Recall'], color='plum', alpha=0.7, label='Recall')
    axes[2].set_xlabel('Score')
    axes[2].set_title('Precision vs Recall')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
else:
    print("\n❌ Tidak ada experiment yang ditemukan!")
    print("💡 Cek manual di: /content/gdrive/MyDrive/yolov5/runs/train/")

## **🔍 CARI MODEL TERBAIK (Accuracy 92%!)**

In [ ]:
# ============================================================================
# AUTOMATIC MODEL FINDER - Cari model dengan accuracy 92%!
# ============================================================================

import os
import pandas as pd
from google.colab import drive

# Mount Google Drive (jika belum)
if not os.path.exists('/content/gdrive'):
    drive.mount('/content/gdrive')

print("="*80)
print("🔍 SEARCHING FOR BEST MODEL (Accuracy 92%)")
print("="*80)

# Possible paths untuk YOLOv5 training results
possible_paths = [
    "/content/gdrive/MyDrive/yolov5/runs/train",
    "/content/yolov5/runs/train",
    "/content/gdrive/My Drive/yolov5/runs/train",
]

all_experiments = []

# Search di semua possible paths
for base_path in possible_paths:
    if os.path.exists(base_path):
        print(f"\n✅ Found: {base_path}")
        
        # List all experiments
        experiments = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
        
        for exp in experiments:
            exp_path = os.path.join(base_path, exp)
            weights_path = os.path.join(exp_path, "weights", "best.pt")
            results_file = os.path.join(exp_path, "results.csv")
            
            # Check if model exists
            if os.path.exists(weights_path):
                model_info = {
                    'path': exp_path,
                    'experiment': exp,
                    'weights_path': weights_path,
                    'size_mb': os.path.getsize(weights_path) / (1024*1024),
                    'exists': True
                }
                
                # Read metrics from results.csv
                if os.path.exists(results_file):
                    try:
                        df = pd.read_csv(results_file)
                        df.columns = df.columns.str.strip()
                        
                        # Get last epoch
                        last = df.iloc[-1]
                        
                        # Extract metrics
                        if 'metrics/precision(B)' in df.columns:
                            model_info['precision'] = last['metrics/precision(B)']
                        if 'metrics/recall(B)' in df.columns:
                            model_info['recall'] = last['metrics/recall(B)']
                        if 'metrics/mAP_0.5(B)' in df.columns:
                            model_info['mAP@0.5'] = last['metrics/mAP_0.5(B)']
                        if 'metrics/mAP_0.5:0.95(B)' in df.columns:
                            model_info['mAP@0.5:0.95'] = last['metrics/mAP_0.5:0.95(B)']
                        
                        # Calculate accuracy (approximate from mAP)
                        if 'mAP@0.5' in model_info:
                            model_info['accuracy'] = model_info['mAP@0.5'] * 100
                            
                    except Exception as e:
                        print(f"⚠️  Warning: Could not read metrics for {exp}: {e}")
                
                all_experiments.append(model_info)

# Display results
if all_experiments:
    print("\n" + "="*80)
    print("📊 ALL TRAINED MODELS FOUND")
    print("="*80)
    
    # Sort by accuracy
    all_experiments_sorted = sorted(all_experiments, 
                                   key=lambda x: x.get('accuracy', 0), 
                                   reverse=True)
    
    for i, model in enumerate(all_experiments_sorted, 1):
        print(f"\n{'🥇' if i==1 else '🥈' if i==2 else '🥉' if i==3 else '📦'} RANK #{i}: {model['experiment']}")
        print(f"   📂 Path: {model['path']}")
        print(f"   💾 Size: {model['size_mb']:.1f} MB")
        
        if 'accuracy' in model:
            acc = model['accuracy']
            status = "🏆 EXCELLENT!" if acc >= 90 else "✅ GOOD" if acc >= 80 else "⚠️  FAIR"
            print(f"   🎯 Accuracy: {acc:.2f}% {status}")
        
        if 'precision' in model:
            print(f"   📊 Precision: {model['precision']:.4f}")
        if 'recall' in model:
            print(f"   📊 Recall: {model['recall']:.4f}")
        if 'mAP@0.5' in model:
            print(f"   📊 mAP@0.5: {model['mAP@0.5']:.4f}")
    
    # Find model dengan accuracy ~92%
    best_models = [m for m in all_experiments_sorted if m.get('accuracy', 0) >= 90]
    
    if best_models:
        print("\n" + "="*80)
        print("🎉 FOUND MODEL(S) WITH ACCURACY ≥ 90%!")
        print("="*80)
        
        for model in best_models:
            print(f"\n🏆 {model['experiment']}")
            print(f"   📂 Full Path: {model['weights_path']}")
            print(f"   🎯 Accuracy: {model.get('accuracy', 0):.2f}%")
            print(f"   📊 mAP@0.5: {model.get('mAP@0.5', 0):.4f}")
            
            # Download command
            print(f"\n   💡 TO DOWNLOAD:")
            print(f"   from google.colab import files")
            print(f"   files.download('{model['weights_path']}')")
    else:
        print("\n⚠️  No model found with accuracy ≥ 90%")
        print("The best model found:")
        best = all_experiments_sorted[0]
        print(f"   {best['experiment']}: {best.get('accuracy', 0):.2f}%")
        
else:
    print("\n❌ NO TRAINED MODELS FOUND!")
    print("\n💡 Possible reasons:")
    print("   1. Models were deleted")
    print("   2. Different Google Drive account")
    print("   3. Models in different path")
    print("\n💡 Manual search:")
    print("   - Check: MyDrive/yolov5/runs/train/")
    print("   - Look for folders: exp, exp2, exp3, etc.")
    print("   - Find: weights/best.pt")

## **🔧 FIX: Generate Accuracy untuk Model exp**

In [ ]:
# ============================================================================
# DEBUG: Cek isi results.csv dan generate metrics yang benar
# ============================================================================

import os
import pandas as pd
from google.colab import drive

# Mount Google Drive first
if not os.path.exists('/content/gdrive'):
    print("🔗 Mounting Google Drive...")
    drive.mount('/content/gdrive')
    print("✅ Drive mounted!\n")

exp_path = "/content/gdrive/MyDrive/yolov5/runs/train/exp"
results_csv = os.path.join(exp_path, "results.csv")

print("="*70)
print("🔍 DEBUGGING MODEL EXP")
print("="*70)

# 1. Cek file apa saja yang ada
print("\n📂 Files in exp folder:")
for file in os.listdir(exp_path):
    if not os.path.isdir(os.path.join(exp_path, file)):
        size = os.path.getsize(os.path.join(exp_path, file)) / 1024
        print(f"   {file} ({size:.1f} KB)")

# 2. Cek results.csv
if os.path.exists(results_csv):
    print("\n✅ results.csv found!")
    
    # Read and display
    try:
        df = pd.read_csv(results_csv)
        print(f"\n📊 Columns: {list(df.columns)}")
        print(f"📊 Total epochs: {len(df)}")
        print(f"\n📋 First 3 rows:")
        print(df.head(3))
        print(f"\n📋 Last 3 rows (FINAL METRICS):")
        print(df.tail(3))
        
        # Try to find accuracy
        print("\n" + "="*70)
        print("🎯 TRYING TO EXTRACT ACCURACY")
        print("="*70)
        
        last_row = df.iloc[-1]
        print(f"\nLast row values:")
        for col in df.columns:
            print(f"   {col}: {last_row[col]}")
            
    except Exception as e:
        print(f"\n❌ Error reading results.csv: {e}")
        print("\n💡 Format might be different. Let's check raw content:")
        
        with open(results_csv, 'r') as f:
            lines = f.readlines()[:10]  # First 10 lines
            for i, line in enumerate(lines):
                print(f"Line {i}: {line.strip()}")
else:
    print("\n❌ results.csv NOT FOUND!")

# 3. Check for results.txt (alternative format)
results_txt = os.path.join(exp_path, "results.txt")
if os.path.exists(results_txt):
    print("\n✅ Found results.txt (alternative format)")
    with open(results_txt, 'r') as f:
        content = f.read()
        print(content[:500])  # First 500 chars

print("\n" + "="*70)
print("💡 NEXT STEP: Run validation to generate accurate metrics")
print("="*70)

In [ ]:
# ============================================================================
# RUN VALIDATION: Generate metrics yang akurat untuk model exp
# ============================================================================

import os
from google.colab import drive

# Mount Google Drive
if not os.path.exists('/content/gdrive'):
    drive.mount('/content/gdrive')

# Change to yolov5 directory
os.chdir('/content')
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git
    
os.chdir('/content/yolov5')

# Path to model and dataset
model_path = "/content/gdrive/MyDrive/yolov5/runs/train/exp/weights/best.pt"
data_yaml = "/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/data_slayer.yaml"

print("="*70)
print("🚀 RUNNING VALIDATION TO GET ACCURATE METRICS")
print("="*70)
print(f"📦 Model: {model_path}")
print(f"📊 Dataset: {data_yaml}")
print(f"⏳ This will take a few minutes...")
print("="*70 + "\n")

# Run validation
!python val.py \
    --data {data_yaml} \
    --weights {model_path} \
    --batch-size 16 \
    --img-size 640 \
    --conf-thres 0.001 \
    --iou-thres 0.6 \
    --save-json \
    --save-txt \
    --save-conf \
    --verbose \
    --plots

print("\n" + "="*70)
print("✅ VALIDATION COMPLETE!")
print("="*70)
print("\n📊 Check runs/val/ folder for:")
print("   - confusion_matrix.png")
print("   - F1_curve.png")
print("   - PR_curve.png")
print("   - P_curve.png")
print("   - R_curve.png")
print("\n💡 Now let's display the results...")

In [ ]:
# ============================================================================
# DISPLAY VALIDATION RESULTS
# ============================================================================

import os
import glob
from IPython.display import display, Image
import matplotlib.pyplot as plt

# Find the latest validation run
val_runs = glob.glob('/content/yolov5/runs/val/*')
if val_runs:
    latest_val = max(val_runs, key=os.path.getmtime)
    
    print("="*70)
    print(f"📊 VALIDATION RESULTS FROM: {latest_val}")
    print("="*70 + "\n")
    
    # Display all plots
    plots = {
        'F1_curve.png': '🎯 F1-CONFIDENCE CURVE',
        'confusion_matrix.png': '🔲 CONFUSION MATRIX',
        'PR_curve.png': '📈 PRECISION-RECALL CURVE',
        'P_curve.png': '📊 PRECISION CURVE',
        'R_curve.png': '📊 RECALL CURVE'
    }
    
    for filename, title in plots.items():
        plot_path = os.path.join(latest_val, filename)
        if os.path.exists(plot_path):
            print(f"\n{title}")
            print("-" * 70)
            display(Image(filename=plot_path))
        else:
            print(f"\n⚠️ {filename} not found")
    
    # Parse results
    results_file = os.path.join(latest_val, "results.csv")
    if os.path.exists(results_file):
        import pandas as pd
        df = pd.read_csv(results_file)
        
        print("\n" + "="*70)
        print("📋 FINAL METRICS")
        print("="*70)
        
        # Get last row (final metrics)
        last = df.iloc[-1]
        
        # Try to extract key metrics
        metrics_to_show = [
            ('mAP@0.5', 'metrics/mAP_0.5'),
            ('mAP@0.5:0.95', 'metrics/mAP_0.5:0.95'),
            ('Precision', 'metrics/precision'),
            ('Recall', 'metrics/recall')
        ]
        
        print("\n🎯 KEY METRICS:")
        for name, col in metrics_to_show:
            if col in df.columns:
                value = last[col]
                print(f"   {name:20s}: {value:.4f} ({value*100:.2f}%)")
        
        # Calculate accuracy estimate (using mAP@0.5)
        if 'metrics/mAP_0.5' in df.columns:
            accuracy = last['metrics/mAP_0.5'] * 100
            print(f"\n{'🏆 ESTIMATED ACCURACY':20s}: {accuracy:.2f}%")
            
            if accuracy >= 92:
                print("\n✅ INI MODEL YANG 92%! 🎉")
            elif accuracy >= 85:
                print("\n✨ Model bagus! (85%+)")
            elif accuracy >= 70:
                print("\n👍 Model cukup baik (70%+)")
            else:
                print("\n⚠️ Model perlu improvement")
    
    print("\n" + "="*70)
    print("💡 If this is the 92% model, download it using Cell 18!")
    print("="*70)
else:
    print("❌ No validation results found. Please run the validation cell first.")

## **📥 DOWNLOAD MODEL TERBAIK ke Local (Windows)**

In [ ]:
# Setelah cell di atas menemukan model terbaik (misal: exp5 dengan 92% accuracy)
# GANTI 'exp5' dengan nama experiment yang ditemukan di cell sebelumnya

from google.colab import files
import shutil

# 🔥 GANTI INI dengan experiment terbaik yang ditemukan!
BEST_EXPERIMENT = "exp5"  # ← GANTI INI!

print("="*70)
print("📥 DOWNLOADING BEST MODEL")
print("="*70)

model_path = f"/content/gdrive/MyDrive/yolov5/runs/train/{BEST_EXPERIMENT}/weights/best.pt"

if os.path.exists(model_path):
    print(f"\n✅ Model found: {model_path}")
    print(f"💾 Size: {os.path.getsize(model_path)/(1024*1024):.1f} MB")
    
    # Copy to easier location first
    easy_path = f"/content/best_model_{BEST_EXPERIMENT}.pt"
    shutil.copy(model_path, easy_path)
    
    print(f"\n🔄 Downloading...")
    files.download(easy_path)
    
    print("\n" + "="*70)
    print("✅ DOWNLOAD COMPLETE!")
    print("="*70)
    print("\n📂 File akan tersimpan di folder Downloads Anda")
    print("\n💡 NEXT STEPS:")
    print(f"   1. Cari file 'best_model_{BEST_EXPERIMENT}.pt' di Downloads")
    print("   2. Copy ke folder SOLVIA:")
    print(f"      C:\\INFORMATIKA\\SEMESTER 5\\Viskom\\AI-Safe\\api\\models\\best.pt")
    print("   3. Start backend: python api/main.py")
    print("   4. Model custom akan auto-load!")
    
else:
    print(f"\n❌ Model not found: {model_path}")
    print("\n💡 Jalankan cell sebelumnya dulu untuk find model!")
    print("💡 Atau cek path manual di Google Drive")

## **Deteksi Menggunakan Model YOLOv5**

In [ ]:
!python detect.py --weights /content/yolov5/runs/train/exp2/weights/best.pt --img 640 --conf 0.25 --source /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/test --save-txt

## **Mengonversi Hasil Deteksi ke Format Submission**

In [ ]:
import os
import pandas as pd

# Path ke folder hasil YOLO (file .txt yang berisi hasil prediksi)
labels_folder = "/content/gdrive/MyDrive/yolov5/runs/detect/exp3/labels"  # Sesuaikan dengan lokasi hasil deteksi
submission_file = "/content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/submission(9).csv"  # Nama file CSV yang akan disimpan

# Inisialisasi list untuk menyimpan hasil
results = []

# Proses setiap file hasil prediksi (file .txt) dalam folder labels
for file in os.listdir(labels_folder):
    if file.endswith(".txt"):
        file_id = file.split(".txt")[0] + ".jpg"  # ID frame (nama file .jpg sesuai dengan .txt)

        with open(os.path.join(labels_folder, file), "r") as f:
            lines = f.readlines()

            # Tentukan label berdasarkan prediksi model
            # Misalnya class 0 untuk 'non_fall' dan class 1 untuk 'fall'
            if any(line.startswith("0") for line in lines):  # Jika ada objek dengan class 0
                label = 0  # Non-fall
            else:
                label = 1  # Fall

            # Simpan hasil ID dan label
            results.append({"id": file_id, "label": label})

# Simpan hasil ke file CSV
df = pd.DataFrame(results)
df.to_csv(submission_file, index=False)

print(f"Submission file saved as {submission_file}")


Submission file saved as /content/gdrive/MyDrive/data-slayer-2-0-machine-learning-competition/submission(9).csv
